In [10]:
import os
import pandas as pd
import json
import warnings
from module.util import report_metrics, find_files
from module.predefined_modality import (
    TextModality, VisionModality, AudioModality, OmniModality, CustomOmniModality
)

warnings.filterwarnings("ignore", category=UserWarning)

# 1. 允许显示所有列（不使用省略号）
pd.set_option('display.max_columns', None)

# 2. 增加列的宽度，防止因太长而换行
pd.set_option('display.max_colwidth', 200)

# 3. 核心：设置总显示宽度为一个很大值，禁止因宽度不够而换行
pd.set_option('display.width', 1000)

# 4. 关键：彻底禁止换行显示
pd.set_option('expand_frame_repr', False)

def get_dataset_stats(dataset_fn, stat_file:str):
    if stat_file is not None and os.path.exists(stat_file):
        with open(stat_file, 'r') as f:
            return json.load(f)
    
    # use python counter util
    from collections import Counter
    dataset = dataset_fn.load_unified(preload=False)
    total_count = len(dataset)
    benign_count = len([item for item in dataset if item['toxicity'] == 0])
    harmful_count = len([item for item in dataset if item['toxicity'] == 1])
    risk_counter = Counter([item['risk'] for item in dataset])
    risk_counts = {risk: risk_counter[risk] for risk in risk_counter}
    assert benign_count == risk_counts.get('BNI', 0), "benign_count != risk_counts.get('BNI', 0)"
    stats = {
        "total_count": total_count,
        "benign_count": benign_count,
        "harmful_count": harmful_count,
        "risk_counts": risk_counts,
    }
    if stat_file is not None:
        with open(stat_file, 'w') as f:
            json.dump(stats, f)
    return stats

def preprocess_results(results, strict:bool):
    # compatible for Qwen3Guard-Gen (strict / loose modes)
    if isinstance(results[0]['pred_toxicity'], str):
        if strict:
            results_strict = [
                {
                    "pred_toxicity": "unsafe" if res['pred_toxicity'].lower() in ['unsafe', 'controversial'] else "safe",
                    "pred_risk": res['pred_risk'],
                    "gt_toxicity": res['gt_toxicity'],
                    "gt_risk": res['gt_risk']
                } for res in results
            ]
            return results_strict
        else:
            results_loose = [
                {
                    "pred_toxicity": "unsafe" if res['pred_toxicity'].lower() == 'unsafe' else "safe",
                    "pred_risk": res['pred_risk'],
                    "gt_toxicity": res['gt_toxicity'],
                    "gt_risk": res['gt_risk']
                } for res in results
            ]
            return results_loose
    return results

def compare_results(compare_models, compare_datasets, compare_metrics, strict=False, metric_tag:str='_fast.'):
    all_metrics = {} # model -> {dataset_1 -> {'accuracy': xxx, ...}}
    all_results = {} # model -> [dataset_metrics]
    all_counts = {}  # model -> (valid_cnt, total_cnt)
    for dataset_name, dataset_fn in compare_datasets:
        dataset_stat = get_dataset_stats(dataset_fn, f"./output/metric.logs/dataset_stats/{dataset_name}.json")
        total_count = dataset_stat["total_count"]
        print(f"Dataset: {dataset_name}")
        print(dataset_stat)

        columns = compare_metrics + ["valid_ratio", "+/-"]
        df = pd.DataFrame(columns=columns)
        for dir in compare_models:
            model_name = dir.split("/")[-1]
            # 根据后缀搜索匹配的文件
            
            json_files = find_files(dir, f"*.{dataset_name}.jsonl")
            if len(json_files) == 0:
                continue
            json_file = None
            if len(json_files) >= 2:
                print(f"Warning: multiple json files found for {model_name}, {json_files}")
                for j in json_files:
                    if metric_tag in j:
                        json_file = j
                        break
                
            if json_file is None: json_file = json_files[0]
            print(f"Json file: {json_file}")
            # read jsonl file into results list
            with open(json_file, 'r') as f:
                results = [json.loads(line) for line in f]
            results = preprocess_results(results, strict)

            if model_name not in all_results:
                all_results[model_name] = []
                all_counts[model_name] = [0, 0] # [valid, total]
                all_metrics[model_name] = {}
            
            valid_count = len(results)
            all_results[model_name].extend(results)
            all_counts[model_name][0] += valid_count
            all_counts[model_name][1] += total_count

            m, gm = report_metrics(results, display=False, dataset_stat=dataset_stat if strict else None)
            all_metrics[model_name][dataset_name] = m
            # write to df
            metrics = [m[k] for k in compare_metrics]
            metrics.append(valid_count / total_count)
            metrics.append(f"{m['count-benign']}/{m['count-harmful']}")
            df.loc[model_name] = metrics
        print(df)
        print("-"*10)
    
    print('='*10)
    for model_name, results in all_results.items(): 
        counts = all_counts[model_name]
        valid, total = counts[0], counts[1]
        print(f"{model_name}: {valid}/{total} = {valid/total:.2%}")
        report_metrics(results, display=True)
    print('='*10)

    for model_name, model_metrics in all_metrics.items():
        print(f"---------{model_name}---------")
        dataset_keys = [item[0] for item in compare_datasets]
        print("\t\t" + " ".join(dataset_keys))
        for m in compare_metrics:
            m_values = [model_metrics[dk][m] for dk in dataset_keys]
            m_values = map(str, [round(v, 4) for v in m_values])
            m_tab = '\t' if len(m)>6 else '\t\t'
            m_str = '\t'.join(m_values)
            print(f"{m}:{m_tab}{m_str}")

    print("==============Done================")
    return all_metrics

report_settings = {
    "General": {
        "Text": [("truthfulQA", TextModality.truthfulQA)],
        "Image": [("mme", VisionModality.mme)],
        "Audio": [("voicebench_alpacaeval", AudioModality.voicebench_alpacaeval)],
        "Video": [("mmbench_video", VisionModality.mmbench_video)],
    },
    "Safety": {
        "Text": [
            ("jbv_redteam_2k", TextModality.jbv_redteam_2k),
            ("beavertails_30k_test", TextModality.beavertails_30k_test),
            ("aegis2_test", TextModality.aegis2_test),
            ("openai_moderation", TextModality.openai_moderation),
            ("wildguardtest", TextModality.wildguardtest),
            ("toxicchat_test", TextModality.toxicchat_test),
        ],
        "Image": [
            # ("vlsafe", VisionModality.vlsafe),
            ("rtvlm", VisionModality.rtvlm),
            ("vlsbench", VisionModality.vlsbench),
            ("vlguard", VisionModality.vlguard),
            ("siuo", VisionModality.siuo),
        ],
        "Audio": [
            ("safebench_ta", AudioModality.safebench_ta),
            ("aiah", AudioModality.aiah),
        ],
        "Video": [("safewatch_real", VisionModality.safewatch_real)],
    },
    "Jailbreak": {
        "Text": [
            ("forbidden_question_dan", TextModality.forbidden_question_dan),
            ("harmbench_contextual", TextModality.harmbench_contextual),
            ("jailbreakbench", TextModality.jailbreakbench),
            ("cipherchat", TextModality.cipherchat),
        ],
        "Image": [
            ("mm_safetybench", VisionModality.mm_safetybench),
            ("jbv_jailbreak_mini", VisionModality.jbv_jailbreak_mini),
            ("figstep", VisionModality.figstep),
            ("mml_hades", VisionModality.mml_hades),
        ],
        "Audio": [
            ("omni_safetybench_dual_ta", AudioModality.omni_safetybench_dual_ta),
            ("ajailbench", AudioModality.ajailbench),
        ],
        "Video": [
            ("omni_safetybench_dual_tv", VisionModality.omni_safetybench_dual_tv),
            ("video_safetybench_ben", VisionModality.video_safetybench_ben),
        ]
    },
    "OmniComb": {
        "Uni": [
            ("omni_safetybench_unimodal_t", TextModality.omni_safetybench_unimodal_t),
            ("omni_safetybench_unimodal_a", AudioModality.omni_safetybench_unimodal_a),
            ("omni_safetybench_unimodal_i", VisionModality.omni_safetybench_unimodal_i),
            ("omni_safetybench_unimodal_v", VisionModality.omni_safetybench_unimodal_v),
            ("safebench_t", TextModality.safebench_t),
        ],
        "Dual": [
            ("omni_safetybench_dual_ta", AudioModality.omni_safetybench_dual_ta),
            ("omni_safetybench_dual_ti", VisionModality.omni_safetybench_dual_ti),
            ("omni_safetybench_dual_tv", VisionModality.omni_safetybench_dual_tv),
            ("safebench_ti", VisionModality.safebench_ti),
            ("safebench_ta", AudioModality.safebench_ta),
        ],
        "Tri": [
            ("omni_safetybench_omni_tia", OmniModality.omni_safetybench_omni_tia),
            ("omni_safetybench_omni_tva", OmniModality.omni_safetybench_omni_tva),
            ("safebench_tia", OmniModality.safebench_tia),
        ]
    },
    "FalseReject": {
        "Text": [("xstest", TextModality.xstest)],
        "Image": [("false_reject_mme", VisionModality.false_reject_mme)],
        "Audio": [("false_reject_alpacaeval", AudioModality.false_reject_alpacaeval)],
        "Video": [("false_reject_mmbench_video", VisionModality.false_reject_mmbench_video)]
    },
    "ModalityBias": {
        "Text": [
            ("omni_custom_T", CustomOmniModality.omni_custom_T),
            ("omni_custom_T_SI", CustomOmniModality.omni_custom_T_SI),
            ("omni_custom_T_SV", CustomOmniModality.omni_custom_T_SV),
            ("omni_custom_T_SA", CustomOmniModality.omni_custom_T_SA),
            ("omni_custom_T_SI_SA", CustomOmniModality.omni_custom_T_SI_SA),
            ("omni_custom_T_SV_SA", CustomOmniModality.omni_custom_T_SV_SA),
            ("omni_custom_T_SI_SV", CustomOmniModality.omni_custom_T_SI_SV),
            ("omni_custom_T_SI_SA_SV", CustomOmniModality.omni_custom_T_SI_SA_SV),
        ],
        "Image": [
            ("omni_custom_I", CustomOmniModality.omni_custom_I),
            ("omni_custom_ST_I", CustomOmniModality.omni_custom_ST_I),
            ("omni_custom_I_SA", CustomOmniModality.omni_custom_I_SA),
            ## ("omni_custom_I_SV", CustomOmniModality.omni_custom_I_SV),
            ("omni_custom_ST_I_SA", CustomOmniModality.omni_custom_ST_I_SA),
            ("omni_custom_ST_I_SV", CustomOmniModality.omni_custom_ST_I_SV),
            ("omni_custom_I_SA_SV", CustomOmniModality.omni_custom_I_SA_SV),
            ("omni_custom_ST_I_SA_SV", CustomOmniModality.omni_custom_ST_I_SA_SV),
        ],
        "Audio": [
            ("omni_custom_A", CustomOmniModality.omni_custom_A),
            ("omni_custom_ST_A", CustomOmniModality.omni_custom_ST_A),
            ("omni_custom_SI_A", CustomOmniModality.omni_custom_SI_A),
            ("omni_custom_SV_A", CustomOmniModality.omni_custom_SV_A),
            ("omni_custom_ST_SI_A", CustomOmniModality.omni_custom_ST_SI_A),
            ("omni_custom_ST_SV_A", CustomOmniModality.omni_custom_ST_SV_A),
            ("omni_custom_SI_A_SV", CustomOmniModality.omni_custom_SI_A_SV),
            ("omni_custom_ST_SI_A_SV", CustomOmniModality.omni_custom_ST_SI_A_SV),
        ],
        "Video": [
            ("omni_custom_V", CustomOmniModality.omni_custom_V),
            ("omni_custom_ST_V", CustomOmniModality.omni_custom_ST_V),
            ## ("omni_custom_SI_V", CustomOmniModality.omni_custom_SI_V),
            ("omni_custom_V_SA", CustomOmniModality.omni_custom_V_SA),
            ("omni_custom_ST_V_SA", CustomOmniModality.omni_custom_ST_V_SA),
            ("omni_custom_ST_SI_V", CustomOmniModality.omni_custom_ST_SI_V),
            ("omni_custom_SI_SA_V", CustomOmniModality.omni_custom_SI_SA_V),
            ("omni_custom_ST_SI_SA_V", CustomOmniModality.omni_custom_ST_SI_SA_V),
        ]
    }
}
def get_compare_datasets(dimension:str, modalities:list[str]):
    if dimension not in report_settings: raise ValueError(f"{dimension} not valid")
    datasets = []
    for modk in modalities:
        datasets.extend(report_settings[dimension][modk])
    return datasets


compare_dirs = [
    # "./output/metric.logs/Qwen2.5-Omni-3B.v3-1",
    # "./output/metric.logs/Omniguard-Qwen25-3B-multimodal-v3-epv3-1.v3-1",
    # "./output/metric.logs/Omniguard-Qwen25-3B-balance-mix-mmv3-epv3-all.v3-1",
    # "./output/metric.logs/Omniguard-Qwen25-3B-balance-mix-mmv3-epv3-all.v3-2",

    # "./output/metric.logs/Qwen2.5-Omni-7B", # zero-shot
    "./output/metric.logs/Qwen2.5-Omni-7B.v3-1", # zero-shot
    # "./output/metric.logs/Omniguard-Qwen25-7B-text-v1-epv3-1",      # normal query only
    # "./output/metric.logs/Omniguard-Qwen25-7B-text-v2-epv3-1",      # +instruct style
    # "./output/metric.logs/Omniguard-Qwen25-7B-text-v3-epv3-1.v3-1", # +jailbreak style
    # "./output/metric.logs/Omniguard-Qwen25-7B-vision-v1-epv3-1",    # normal query only
    # "./output/metric.logs/Omniguard-Qwen25-7B-vision-v3-epv3-1",    # +jailbreak style
    # "./output/metric.logs/Omniguard-Qwen25-7B-audio-v3-epv3-1",
    # "./output/metric.logs/Omniguard-Qwen25-7B-text-audio-v3-epv3-1",
    # "./output/metric.logs/Omniguard-Qwen25-7B-text-vision-v3-epv3-1",
    # "./output/metric.logs/Omniguard-Qwen25-7B-vision-audio-v3-epv3-1",

    # "./output/metric.logs/Omniguard-Qwen25-7B-text-v3-epv3-1-full.v3-1",          # Full SFT
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-1-full.v3-1",    # Full SFT
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-contra-v3-epv3-1.v3-1",

    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-1",         # basic
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-1-a.v3-1",
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-1-b.v3-1",

    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-all.v3-1",  # SAE
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-all.v3-2",  # SAE

    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-aligner1-epv3-1",      # balance only
    # "./output/metric.logs/Omniguard-Qwen25-7B-judge-mmv3-epv3-1",            # balance -> basic
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-1",      # MRB
    
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-2",
    # "./output/metric.logs/Omniguard-Qwen25-7B-judge-mmv3-epv3-2a",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-2",
    
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-3",
    # "./output/metric.logs/Omniguard-Qwen25-7B-judge-mmv3-epv3-3",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-3",
    
    # "./output/metric.logs/Omniguard-Qwen25-7B-multimodal-v3-epv3-4",
    # "./output/metric.logs/Omniguard-Qwen25-7B-judge-mmv3-epv3-4",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-4",

    "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-1",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-1-balance",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-2",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-3",
    # "./output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-4",

    # "./output/metric.logs/Qwen3-Omni-30B-A3B-Instruct", # zero-shot
    # "./output/metric.logs/Omniguard-Qwen3-30B-text-v3-epv3-1.v3-1",
    # "./output/metric.logs/Omniguard-Qwen3-30B-multimodal-v3-epv3-1.v3-1",
    # "./output/metric.logs/Omniguard-Qwen3-30B-multimodal-v3-epv3-2.v3-2",
    # "./output/metric.logs/Omniguard-Qwen3-30B-multimodal-v3-epv3-3.v3-3",
    # "./output/metric.logs/Omniguard-Qwen3-30B-multimodal-v3-epv3-4.v3-4",
    # "./output/metric.logs/Omniguard-Qwen3-30B-multimodal-v3-epv3-all.v3-1",
    # "./output/metric.logs/Omniguard-Qwen3-30B-multimodal-v3-epv3-all.v3-2",

    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-aligner1-epv3-1.v3-1",
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-1.v3-1",  # MRB
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-b.v3-1", # partly used 
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-b.v3-2",
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-b.v3-3",
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-b.v3-4",
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-a.v3-1", # main
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-a.v3-2",
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-a.v3-3",
    # "./output/metric.logs/Omniguard-Qwen3-30B-balance-mix-mmv3-epv3-all-a.v3-4",

    # "./output/metric.logs/MiniCPM-o-4_5.v3-1",
    # "./output/metric.logs/Omniguard-MiniCPM-multimodal-v3-epv3-1.v3-1",
    # "./output/metric.logs/Omniguard-MiniCPM-balance-mix-mmv3-epv3-all.v3-1",
    # "./output/metric.logs/Omniguard-MiniCPM-balance-mix-mmv3-epv3-all.v3-2",
    
    # "./output/metric.logs/Phi-4-multimodal-instruct.v3-1",
    # "./output/metric.logs/Omniguard-Phi-multimodal-v3-epv3-1.v3-1",
    # "./output/metric.logs/Omniguard-Phi-balance-mix-mmv3-epv3-all.v3-1",
    # "./output/metric.logs/Omniguard-Phi-balance-mix-mmv3-epv3-all.v3-2",

    "./output/metric.logs/Omniguard-7B",
    # "./output/metric.logs/Qwen3Guard-Gen-8B",
    # "./output/metric.logs/Llama-Guard-3-8B",
    # "./output/metric.logs/Llama-Guard-3-11B-Vision"
]

dimension = "OmniComb" # General,Safety,Jailbreak,FalseReject,ModalityBias,OmniComb
# modalities = ["Text", "Image", "Audio", "Video"]
modalities = ["Tri"]
# modalities = ["Text", "Image", "Audio"]
# modalities = ["Image",]
compare_metrics = ["accuracy", "precision", "recall", "f1", "fpr"]
# compare_datasets = get_compare_datasets("General", ["Text", "Image", "Audio", "Video"])
compare_datasets = get_compare_datasets(dimension, modalities)
all_metrics = compare_results(compare_dirs, compare_datasets, compare_metrics, strict=True)

if dimension == 'ModalityBias':
    dataset_keys = [item[0] for item in compare_datasets]
    for model_key, metrics in all_metrics.items():
        mbi_list = []
        for metric_base, index_name in [('accuracy', 'MSI@ACC'), ('recall', 'MSI@REC'), ('f1', 'MSI@F1')]:
            a = metrics[dataset_keys[0]][metric_base]
            bs = [metrics[dk][metric_base] for dk in dataset_keys[1:]]
            b = sum(bs)/len(bs)
            aMSI = a - b
            nMSI = max(a - b, 0) / (a + 1e-9)
            mV = (a+sum(bs))/(1+len(bs))
            print(model_key, f"{metric_base}(avg)", round(mV, 4), \
                    f"{metric_base}(uni)", round(a, 4), \
                    f"{metric_base}(conflict)", round(b, 4), \
                index_name, round(nMSI, 4))
        print("-----------")


Dataset: omni_safetybench_omni_tia
{'total_count': 5832, 'benign_count': 0, 'harmful_count': 5832, 'risk_counts': {'EH': 732, 'FA': 924, 'HS': 978, 'IA': 582, 'CS': 264, 'PH': 864, 'PV': 834, 'SC': 654}}
Json file: output/metric.logs/Qwen2.5-Omni-7B.v3-1/omniguard_sft_fast.omni_safetybench_omni_tia.jsonl
Json file: output/metric.logs/Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-1/omniguard_sft_fast.omni_safetybench_omni_tia.jsonl
Json file: output/metric.logs/Omniguard-7B/omniguard_omni.omni_safetybench_omni_tia.jsonl
                                                    accuracy  precision    recall        f1  fpr  valid_ratio     +/-
Qwen2.5-Omni-7B.v3-1                                0.933299        1.0  0.933299  0.965499  NaN     0.999829  0/5832
Omniguard-Qwen25-7B-balance-mix-mmv3-epv3-all.v3-1  0.992970        1.0  0.992970  0.996473  NaN     0.999829  0/5832
Omniguard-7B                                        0.905350        1.0  0.905350  0.950324  NaN     0.999657  0/5832
